# 01 — Exploratory Data Analysis

**Rule:** All EDA is performed on the train split ONLY.  
The test set is locked at the bottom of this notebook and never touched until `05_evaluation.ipynb`.

In [ ]:
!pip install -q lightgbm shap missingno matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)

In [ ]:
# Load raw data — Kaggle path
trans = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
ident = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')

df = trans.merge(ident, on='TransactionID', how='left')
print(f'Shape after merge: {df.shape}')
print(f'Fraud rate: {df.isFraud.mean():.4%}')

## Time-based split — lock test set NOW

In [ ]:
df = df.sort_values('TransactionDT').reset_index(drop=True)
n = len(df)

train = df.iloc[:int(n * 0.70)].copy()
val   = df.iloc[int(n * 0.70):int(n * 0.85)].copy()
test  = df.iloc[int(n * 0.85):].copy()

print(f'Train: {len(train):,} rows | fraud: {train.isFraud.mean():.4%}')
print(f'Val:   {len(val):,} rows | fraud: {val.isFraud.mean():.4%}')
print(f'Test:  {len(test):,} rows | fraud: {test.isFraud.mean():.4%}')

# Lock test — save and do not open again until 05_evaluation
test.to_parquet('/kaggle/working/test_locked.parquet', index=False)
val.to_parquet('/kaggle/working/val.parquet', index=False)
train.to_parquet('/kaggle/working/train.parquet', index=False)

print('\nSplits saved. Test set is now LOCKED.')

## Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = train.isFraud.value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('Class distribution (absolute)')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'],
            colors=['steelblue', 'crimson'], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class distribution (%)')

plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150)
plt.show()

## Transaction amount distribution by class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, 'steelblue'), (1, 'crimson')]:
    name = 'Legitimate' if label == 0 else 'Fraud'
    axes[0].hist(train[train.isFraud == label]['TransactionAmt'],
                 bins=100, alpha=0.6, color=color, label=name)
    axes[1].hist(np.log1p(train[train.isFraud == label]['TransactionAmt']),
                 bins=100, alpha=0.6, color=color, label=name)

axes[0].set_title('Transaction Amount (raw)')
axes[0].legend()
axes[1].set_title('Transaction Amount (log1p)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/amount_distribution.png', dpi=150)
plt.show()

## Temporal patterns — fraud by hour and day

In [ ]:
# TransactionDT is seconds offset from a reference point
START_DATE = pd.Timestamp('2017-12-01')
train['datetime'] = START_DATE + pd.to_timedelta(train['TransactionDT'], unit='s')
train['hour'] = train['datetime'].dt.hour
train['dow']  = train['datetime'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly_fraud = train.groupby('hour')['isFraud'].mean()
axes[0].bar(hourly_fraud.index, hourly_fraud.values, color='crimson', alpha=0.7)
axes[0].set_title('Fraud rate by hour of day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Fraud rate')

day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
daily_fraud = train.groupby('dow')['isFraud'].mean()
axes[1].bar([day_names[d] for d in daily_fraud.index], daily_fraud.values, color='steelblue', alpha=0.7)
axes[1].set_title('Fraud rate by day of week')

plt.tight_layout()
plt.savefig('/kaggle/working/temporal_patterns.png', dpi=150)
plt.show()

## Missing value analysis

In [ ]:
missing_pct = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)
high_missing = missing_pct[missing_pct > 50]

print(f'Columns with >50% missing: {len(high_missing)}')
print(f'\nTop 20 by missing %:')
print(missing_pct.head(20))

# Save columns to drop
cols_to_drop = list(high_missing.index)
import json
with open('/kaggle/working/cols_to_drop.json', 'w') as f:
    json.dump(cols_to_drop, f)

print(f'\nSaved {len(cols_to_drop)} columns to drop to cols_to_drop.json')

## Key findings summary

In [ ]:
print('=== EDA SUMMARY ===')
print(f'Total training transactions: {len(train):,}')
print(f'Fraud rate: {train.isFraud.mean():.4%} — severe class imbalance')
print(f'Median legitimate amount: ${train[train.isFraud==0]["TransactionAmt"].median():.2f}')
print(f'Median fraud amount: ${train[train.isFraud==1]["TransactionAmt"].median():.2f}')
print(f'Columns with >50% missing (will drop): {len(cols_to_drop)}')
print()
print('Key observations:')
print('1. Fraud transactions tend to have higher amounts')
print('2. Night hours (0-5am) show elevated fraud rate')
print('3. Severe class imbalance requires careful metric selection and sampling strategy')
print('4. Many V-columns have >50% missing — feature selection critical')